## Vuelos enriquecidos 

In [0]:
%sql
CREATE OR REPLACE TABLE aerolinea.gold.vuelos_enriquecidos
USING DELTA AS
SELECT v.vuelo_id, v.fecha,
       CAST(date_trunc('MONTH', v.fecha) AS DATE) AS mes,
       v.aeronave_id, a.modelo, a.fabricante, a.anio_fabricacion,
       YEAR(v.fecha) - a.anio_fabricacion AS edad_aeronave,
       v.origen_id, o.nombre AS origen_nombre, o.ciudad AS origen_ciudad, o.pais AS origen_pais,
       v.destino_id, d.nombre AS destino_nombre, d.ciudad AS destino_ciudad, d.pais AS destino_pais,
       v.estado, v.duracion_min, v.estado <> 'cancelado' AS operado,
       v.ruta_por_revisar, v.archivo_origen
FROM aerolinea.silver.vuelos v
LEFT JOIN aerolinea.silver.aeronaves a ON v.aeronave_id = a.aeronave_id
LEFT JOIN aerolinea.silver.aeropuertos o ON v.origen_id = o.aeropuerto_id
LEFT JOIN aerolinea.silver.aeropuertos d ON v.destino_id = d.aeropuerto_id

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM aerolinea.gold.vuelos_enriquecidos ORDER BY vuelo_id LIMIT 10

vuelo_id,fecha,mes,aeronave_id,modelo,fabricante,anio_fabricacion,edad_aeronave,origen_id,origen_nombre,origen_ciudad,origen_pais,destino_id,destino_nombre,destino_ciudad,destino_pais,estado,duracion_min,operado,ruta_por_revisar,archivo_origen
V0001,2024-05-21,2024-05-01,A005,B777,Boeing,2007,17,MAD,Adolfo Suárez Madrid-Barajas,Madrid,España,JFK,John F. Kennedy International,Nueva York,EEUU,a_tiempo,288,true,false,vuelos_diarios.csv
V0002,2024-05-05,2024-05-01,A002,B737,Boeing,2013,11,MAD,Adolfo Suárez Madrid-Barajas,Madrid,España,LHR,Heathrow,Londres,Reino Unido,cancelado,492,false,false,vuelos_diarios.csv
V0003,2024-05-02,2024-05-01,A004,B737,Boeing,2013,11,MAD,Adolfo Suárez Madrid-Barajas,Madrid,España,JFK,John F. Kennedy International,Nueva York,EEUU,a_tiempo,577,true,false,vuelos_diarios.csv
V0004,2024-05-20,2024-05-01,A004,B737,Boeing,2013,11,MAD,Adolfo Suárez Madrid-Barajas,Madrid,España,LHR,Heathrow,Londres,Reino Unido,cancelado,618,false,false,vuelos_diarios.csv
V0005,2024-05-14,2024-05-01,A010,B737,Boeing,2012,12,BCN,Josep Tarradellas Barcelona-El Prat,Barcelona,España,BCN,Josep Tarradellas Barcelona-El Prat,Barcelona,España,retrasado,66,true,true,vuelos_diarios.csv
V0006,2024-05-25,2024-05-01,A006,A350,Airbus,2012,12,BCN,Josep Tarradellas Barcelona-El Prat,Barcelona,España,BCN,Josep Tarradellas Barcelona-El Prat,Barcelona,España,retrasado,219,true,true,vuelos_diarios.csv
V0007,2024-05-07,2024-05-01,A002,B737,Boeing,2013,11,JFK,John F. Kennedy International,Nueva York,EEUU,JFK,John F. Kennedy International,Nueva York,EEUU,retrasado,159,true,true,vuelos_diarios.csv
V0008,2024-05-12,2024-05-01,A005,B777,Boeing,2007,17,JFK,John F. Kennedy International,Nueva York,EEUU,LHR,Heathrow,Londres,Reino Unido,a_tiempo,530,true,false,vuelos_diarios.csv
V0009,2024-05-18,2024-05-01,A002,B737,Boeing,2013,11,MAD,Adolfo Suárez Madrid-Barajas,Madrid,España,BCN,Josep Tarradellas Barcelona-El Prat,Barcelona,España,cancelado,360,false,false,vuelos_diarios.csv
V0010,2024-05-27,2024-05-01,A010,B737,Boeing,2012,12,CDG,Charles de Gaulle,París,Francia,CDG,Charles de Gaulle,París,Francia,a_tiempo,131,true,true,vuelos_diarios.csv


## Mantenimiento enriquecido

In [0]:
%sql
CREATE OR REPLACE TABLE aerolinea.gold.mantenimientos_enriquecidos
USING DELTA AS
SELECT m.mantenimiento_id, m.fecha,
       CAST(date_trunc('MONTH', m.fecha) AS DATE) AS mes,
       m.aeronave_id, a.modelo, a.fabricante, a.anio_fabricacion,
       YEAR(m.fecha) - a.anio_fabricacion AS edad_aeronave,
       m.tipo, m.costo_usd, m.duracion_hr, m.archivo_origen
FROM aerolinea.silver.mantenimientos m
LEFT JOIN aerolinea.silver.aeronaves a ON m.aeronave_id = a.aeronave_id

num_affected_rows,num_inserted_rows


**Comprobación**

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM aerolinea.gold.vuelos_enriquecidos) AS vuelos_enriquecidos,
  (SELECT COUNT(*) FROM aerolinea.silver.vuelos) AS vuelos_silver,
  (SELECT COUNT(*) FROM aerolinea.gold.mantenimientos_enriquecidos) AS mantenimientos_enriquecidos,
  (SELECT COUNT(*) FROM aerolinea.silver.mantenimientos) AS mantenimientos_silver,
  CASE WHEN (SELECT COUNT(*) FROM aerolinea.gold.vuelos_enriquecidos) = (SELECT COUNT(*) FROM aerolinea.silver.vuelos)
       THEN 'OK' ELSE 'MISMATCH: La unión multiplicó o perdió vuelos.' END AS validacion_vuelos,
  CASE WHEN (SELECT COUNT(*) FROM aerolinea.gold.mantenimientos_enriquecidos) = (SELECT COUNT(*) FROM aerolinea.silver.mantenimientos)
       THEN 'OK' ELSE 'MISMATCH: La unión multiplicó o perdió intervenciones.' END AS validacion_mantenimientos

vuelos_enriquecidos,vuelos_silver,mantenimientos_enriquecidos,mantenimientos_silver,validacion_vuelos,validacion_mantenimientos
100,100,100,100,OK,OK


## Puntualidad x grupo

In [0]:
%sql
CREATE OR REPLACE TABLE aerolinea.gold.kpi_puntualidad_vuelos
USING DELTA AS
SELECT modelo, fabricante, origen_pais, mes,
       COUNT(*) AS total_vuelos,
       SUM(CASE WHEN operado THEN 1 ELSE 0 END) AS vuelos_operados,
       SUM(CASE WHEN estado = 'a_tiempo' THEN 1 ELSE 0 END) AS vuelos_a_tiempo,
       SUM(CASE WHEN estado = 'retrasado' THEN 1 ELSE 0 END) AS vuelos_retrasados,
       SUM(CASE WHEN estado = 'cancelado' THEN 1 ELSE 0 END) AS vuelos_cancelados,
       ROUND(100.0 * SUM(CASE WHEN estado = 'a_tiempo' THEN 1 ELSE 0 END)
         / NULLIF(SUM(CASE WHEN operado THEN 1 ELSE 0 END), 0), 2) AS puntualidad_operados_pct,
       ROUND(100.0 * SUM(CASE WHEN estado = 'cancelado' THEN 1 ELSE 0 END)
         / COUNT(*), 2) AS cancelacion_pct,
       ROUND(AVG(CASE WHEN operado THEN duracion_min END), 2) AS duracion_promedio_operados_min,
       SUM(CASE WHEN ruta_por_revisar THEN 1 ELSE 0 END) AS rutas_por_revisar
FROM aerolinea.gold.vuelos_enriquecidos
GROUP BY modelo, fabricante, origen_pais, mes

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM aerolinea.gold.kpi_puntualidad_vuelos ORDER BY modelo, origen_pais

modelo,fabricante,origen_pais,mes,total_vuelos,vuelos_operados,vuelos_a_tiempo,vuelos_retrasados,vuelos_cancelados,puntualidad_operados_pct,cancelacion_pct,duracion_promedio_operados_min,rutas_por_revisar
A320,Airbus,EEUU,2024-05-01,1,1,1,0,0,100.00,0.00,345.0,1
A320,Airbus,España,2024-05-01,6,5,2,3,1,40.00,16.67,534.6,0
A320,Airbus,Francia,2024-05-01,2,2,0,2,0,0.00,0.00,396.0,0
A320,Airbus,Reino Unido,2024-05-01,4,3,3,0,1,100.00,25.00,363.67,0
A350,Airbus,EEUU,2024-05-01,6,4,2,2,2,50.00,33.33,362.5,0
A350,Airbus,España,2024-05-01,12,9,3,6,3,33.33,25.00,342.0,4
A350,Airbus,Francia,2024-05-01,6,4,2,2,2,50.00,33.33,427.75,1
A350,Airbus,Reino Unido,2024-05-01,4,3,1,2,1,33.33,25.00,344.33,1
B737,Boeing,EEUU,2024-05-01,4,3,2,1,1,66.67,25.00,262.0,2
B737,Boeing,España,2024-05-01,17,10,9,1,7,90.00,41.18,344.0,5


Databricks visualization. Run in Databricks to view.

Puntualidad **global**

In [0]:
%sql
SELECT SUM(total_vuelos) AS vuelos,
       SUM(vuelos_operados) AS operados,
       SUM(vuelos_a_tiempo) AS a_tiempo,
       SUM(vuelos_cancelados) AS cancelados,
       ROUND(100.0 * SUM(vuelos_a_tiempo) / NULLIF(SUM(vuelos_operados), 0), 2) AS puntualidad_pct,
       ROUND(100.0 * SUM(vuelos_cancelados) / SUM(total_vuelos), 2) AS cancelacion_pct
FROM aerolinea.gold.kpi_puntualidad_vuelos

vuelos,operados,a_tiempo,cancelados,puntualidad_pct,cancelacion_pct
100,69,40,31,57.97,31.00


## Costos del Mantenimiento

In [0]:
%sql
CREATE OR REPLACE TABLE aerolinea.gold.kpi_costo_mantenimiento
USING DELTA AS
SELECT aeronave_id, modelo, fabricante, tipo, mes,
       COUNT(*) AS total_mantenimientos,
       SUM(costo_usd) AS costo_total_usd,
       ROUND(AVG(costo_usd), 2) AS costo_promedio_usd,
       SUM(duracion_hr) AS duracion_total_hr,
       ROUND(AVG(duracion_hr), 2) AS duracion_promedio_hr
FROM aerolinea.gold.mantenimientos_enriquecidos
GROUP BY aeronave_id, modelo, fabricante, tipo, mes

num_affected_rows,num_inserted_rows


## Gráfico de Costos

In [0]:
%sql
SELECT tipo,
       SUM(total_mantenimientos) AS intervenciones,
       SUM(costo_total_usd) AS costo_total_usd
FROM aerolinea.gold.kpi_costo_mantenimiento
GROUP BY tipo
ORDER BY costo_total_usd DESC

tipo,intervenciones,costo_total_usd
correctivo,36,987163.77
inspección,31,861286.76
preventivo,33,740904.85


Databricks visualization. Run in Databricks to view.

##  Actividad y costo en el mismo nivel de detalle

In [0]:
%sql

CREATE OR REPLACE TABLE aerolinea.gold.resumen_aeronave_mes
USING DELTA AS
WITH actividad AS (
  SELECT aeronave_id, mes, COUNT(*) AS total_vuelos,
         SUM(CASE WHEN operado THEN 1 ELSE 0 END) AS vuelos_operados
  FROM aerolinea.gold.vuelos_enriquecidos GROUP BY aeronave_id, mes
), mantenimiento AS (
  SELECT aeronave_id, mes, COUNT(*) AS total_mantenimientos,
         SUM(costo_usd) AS costo_total_usd
  FROM aerolinea.gold.mantenimientos_enriquecidos GROUP BY aeronave_id, mes
)
SELECT COALESCE(v.aeronave_id, m.aeronave_id) AS aeronave_id,
       COALESCE(v.mes, m.mes) AS mes,
       COALESCE(v.total_vuelos, 0) AS total_vuelos,
       COALESCE(v.vuelos_operados, 0) AS vuelos_operados,
       COALESCE(m.total_mantenimientos, 0) AS total_mantenimientos,
       COALESCE(m.costo_total_usd, 0) AS costo_total_usd
FROM actividad v FULL OUTER JOIN mantenimiento m
  ON v.aeronave_id = m.aeronave_id AND v.mes = m.mes

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM aerolinea.gold.resumen_aeronave_mes ORDER BY aeronave_id, mes

aeronave_id,mes,total_vuelos,vuelos_operados,total_mantenimientos,costo_total_usd
A001,2024-05-01,11,8,13,306309.36
A002,2024-05-01,10,6,7,180100.27
A003,2024-05-01,10,6,10,275186.84
A004,2024-05-01,13,6,6,129149.13
A005,2024-05-01,6,5,10,225937.22
A006,2024-05-01,11,9,11,301796.52
A007,2024-05-01,9,6,12,358043.88
A008,2024-05-01,8,5,7,203249.28
A009,2024-05-01,13,11,12,240600.47
A010,2024-05-01,9,7,12,368982.41


## Construcción base para ML

In [0]:
%sql
CREATE OR REPLACE TABLE aerolinea.gold.base_ml_mantenimiento
USING DELTA AS
SELECT mantenimiento_id, fecha, aeronave_id,
       tipo, modelo, fabricante, edad_aeronave, costo_usd
FROM aerolinea.gold.mantenimientos_enriquecidos

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM aerolinea.gold.base_ml_mantenimiento ORDER BY mantenimiento_id LIMIT 10

mantenimiento_id,fecha,aeronave_id,tipo,modelo,fabricante,edad_aeronave,costo_usd
M0001,2024-05-29,A001,inspección,B777,Boeing,11,13189.29
M0002,2024-05-27,A003,correctivo,E190,Embraer,4,36860.78
M0003,2024-05-12,A010,correctivo,B737,Boeing,12,25147.00
M0004,2024-05-23,A002,inspección,B737,Boeing,11,44255.92
M0005,2024-05-21,A007,correctivo,A350,Airbus,9,14791.71
M0006,2024-05-30,A010,inspección,B737,Boeing,12,31080.70
M0007,2024-05-29,A005,correctivo,B777,Boeing,17,37821.40
M0008,2024-05-03,A005,correctivo,B777,Boeing,17,7613.00
M0009,2024-05-15,A006,inspección,A350,Airbus,12,6587.11
M0010,2024-05-04,A006,preventivo,A350,Airbus,12,11258.27
